remember: GPU T4 X2 and on Internet in Session options
        
          Create CLOUDFLARE_TUNNEL_TOKEN or (NGROK_AUTHTOKEN + NGROK_DOMAIN) if you want to use a fixed domain (Add-ons -> Secrets -> Add a new secret)

docs: https://github.com/Niko1221/Strata

strata repo: https://github.com/Niko1221/Strata (run ./setup.sh, server on port 8080)

## Optional: API key for the Strata server

The Install cell reads a Kaggle secret and passes it to setup as `--api-key`:

  Label: API_KEY      Value: (the key clients must send)

If the secret is not set, the `--api-key` flag is skipped and the server runs with no API key.
With a key: requests need `Authorization: Bearer <key>` or `x-api-key: <key>` (`/health` stays open).

## Optional: Custom Google API Client ID/Secret (avoid rclone rate limits)

rclone by default uses shared OAuth app -> may hit Google rate limits (429 Too Many Requests) when many users online.

To use YOUR OWN Google API credentials:

1. Go to https://console.cloud.google.com/
2. Create new project (any name, e.g. "rclone-backup")
3. APIs & Services -> Library -> search "Google Drive API" -> Enable
4. APIs & Services -> OAuth consent screen:
   - User type: External -> Create
   - App name: anything (e.g. "My Rclone Backup")
   - User support email: your email
   - Developer contact: your email
   - Save and Continue through scopes (no scopes needed here, rclone will request them in authorize)
   - Add yourself to "Test users" section (your email)
   - Save (app stays in "Testing" status which is fine for personal use)
5. APIs & Services -> Credentials:
   - Create Credentials -> OAuth client ID
   - Application type: **Desktop app** (important: not Web app)
   - Name: anything
   - Create -> copy **Client ID** + **Client Secret**
6. Back in Kaggle notebook: Add-ons -> Secrets -> Add new secret (2 secrets):
   - Label: `GDRIVE_CLIENT_ID`        Value: your Client ID
   - Label: `GDRIVE_CLIENT_SECRET`    Value: your Client Secret
7. Cell 2 prompt "Use custom Google API Client ID/Secret? (y/N)" -> answer "y"

Notes:
- First time switching from rclone default to custom credentials: existing `RCLONE_DRIVE_TOKEN` will be auto-detected as stale and re-auth will be triggered. Just follow the OAuth link once more.
- App in "Testing" status requires your Google account to be in "Test users" list. Publish is optional.
- Custom credentials have their own Google Drive API quota -> avoids shared rclone rate limits.

# Setup

In [ ]:
# ================= GOOGLE DRIVE CONFIG =================
REMOTE = "gdrive"
DRIVE_BACKUP_DIR = "Strata"

# (local path, Drive subfolder, rclone --filter rules; list rong = include all)
# Whitelist: chi backup phan KHONG the tao lai duoc moi session.
# Git source, .venv, third_party, log... de git clone + setup tai lai tu dong.
SOURCES = [
    ("/root/Strata", "", [
        "- engine/.previous/**",
        "+ engine/**",
        "+ strata-*.json",
        "+ run-*",
        "+ data/**",
        "- **",
    ]),
    ("/root/.config/strata", "config", []),
]

SYNC_INTERVAL = 300  # seconds between rclone sync

# ================= SETUP CHOICES =================
drive = input("Configure Google Drive backup? (y/N): ").strip().lower()
CONFIGURE_DRIVE = drive == "y"

# ================= CUSTOM GOOGLE API (optional) =================
use_custom = input("Use custom Google API Client ID/Secret? (y/N): ").strip().lower()
USE_CUSTOM_GDRIVE_API = use_custom == "y"
if USE_CUSTOM_GDRIVE_API:
    print("  -> Using custom Client ID/Secret (from Kaggle Secrets GDRIVE_CLIENT_ID/SECRET)")
    print("  -> Required: see setup guide in Cell 0 above")
else:
    print("  -> Using rclone default OAuth (may hit rate limits on busy days)")

print("Tunnel mode:")
print("  1) Quick Tunnel (Cloudflare - khong can cau hinh)")
print("  2) Named Tunnel (can CLOUDFLARE_TUNNEL_TOKEN)")
print("  3) Ngrok (can NGROK_AUTHTOKEN + NGROK_DOMAIN)")
choice = input("Chon (1/2/3): ").strip()
TUNNEL_MODE = {"1": "quick", "2": "named", "3": "ngrok"}.get(choice, "quick")
print(f"Tunnel mode: {TUNNEL_MODE}")

# ================= STRATA PORT + TUNNEL TOKEN (Named only) =================
STRATA_PORT = "8080"
TUNNEL_NUM = "1"
TUNNEL_TOKEN_LABEL = "CLOUDFLARE_TUNNEL_TOKEN"
if TUNNEL_MODE == "named":
    use_custom_port = input("Dung custom port cho Strata? (y/N): ").strip().lower()
    if use_custom_port == "y":
        raw_port = input("Nhap custom port (1024-65535, Enter = 8081): ").strip()
        try:
            p = int(raw_port) if raw_port else 8081
            if 1024 <= p <= 65535 and p != 53682:
                STRATA_PORT = str(p)
            else:
                print(f"  -> Port {raw_port!r} khong hop le (hoac trung 53682 cua rclone OAuth), dung 8081")
                STRATA_PORT = "8081"
        except ValueError:
            print(f"  -> Input {raw_port!r} khong phai so, dung 8081")
            STRATA_PORT = "8081"
        raw_num = input("Dung CLOUDFLARE_TUNNEL_TOKEN so may? (1=goc, 2, 3...; Enter = 2): ").strip()
        TUNNEL_NUM = raw_num if raw_num else "2"
        if TUNNEL_NUM == "1":
            TUNNEL_TOKEN_LABEL = "CLOUDFLARE_TUNNEL_TOKEN"
        else:
            TUNNEL_TOKEN_LABEL = f"CLOUDFLARE_TUNNEL_TOKEN-{TUNNEL_NUM}"
        print(f"Tunnel token: {TUNNEL_TOKEN_LABEL}")
print(f"Strata port: {STRATA_PORT}")

In [ ]:

!git clone https://github.com/Niko1221/Strata.git /root/Strata
%cd /root/Strata
!python -m pip install -q virtualenv
!python -m virtualenv .venv
!.venv/bin/python -m pip install -q wrapt
!/root/Strata/.venv/bin/python -c "import wrapt; print('wrapt Version:', wrapt.__version__)"
CUDA_PATH = "/usr/local/cuda"

In [ ]:
# %%bash
# set -euo pipefail

# CUDA_DIR="/usr/local/cuda-13.0"

# # Nếu CUDA 13 chưa được cài thì mới thêm repository và cài
# if [ ! -x "$CUDA_DIR/bin/nvcc" ]; then
#     . /etc/os-release

#     case "$VERSION_ID" in
#         "24.04") DISTRO="ubuntu2404" ;;
#         "22.04") DISTRO="ubuntu2204" ;;
#         *)
#             echo "Ubuntu $VERSION_ID khong duoc ho tro."
#             exit 1
#             ;;
#     esac

#     echo "Removing conflicting CUDA repositories..."

#     find /etc/apt/sources.list.d -maxdepth 1 -type f \
#         \( -iname "*cuda*.list" -o -iname "*cuda*.sources" \) \
#         -delete

#     if [ -f /etc/apt/sources.list ]; then
#         sed -i '\|developer.download.nvidia.com/compute/cuda|d' \
#             /etc/apt/sources.list
#     fi

#     echo "Installing NVIDIA CUDA repository..."

#     wget -q \
#         "https://developer.download.nvidia.com/compute/cuda/repos/${DISTRO}/x86_64/cuda-keyring_1.1-1_all.deb" \
#         -O /tmp/cuda-keyring.deb

#     dpkg -i /tmp/cuda-keyring.deb

#     apt-get update

#     echo "Installing CUDA Toolkit 13.0..."

#     DEBIAN_FRONTEND=noninteractive \
#         apt-get install -y cuda-toolkit-13-0
# else
#     echo "CUDA Toolkit 13.0 is already installed."
# fi

# echo "Setting CUDA 13.0 as default..."

# ln -sfn "$CUDA_DIR" /usr/local/cuda

# printf '%s\n' "$CUDA_DIR/lib64" \
#     > /etc/ld.so.conf.d/cuda-13-0.conf

# ldconfig

# echo
# echo "CUDA symlink:"
# readlink -f /usr/local/cuda

# echo
# echo "NVCC version:"
# /usr/local/cuda/bin/nvcc --version

# echo
# echo "CUDA Toolkit 13.0 setup completed."
# CUDA_PATH = "/usr/local/cuda-13.0"

In [ ]:
!readlink -f /usr/local/cuda
!which nvcc
!nvcc --version
!nvidia-smi


In [ ]:
import os

os.environ["CUDA_HOME"] = CUDA_PATH
os.environ["CUDAToolkit_ROOT"] = CUDA_PATH
os.environ["CUDACXX"] = f"{CUDA_PATH}/bin/nvcc"
os.environ["CUDA_DRIVER_LIBRARY"] = "/usr/local/nvidia/lib64/libcuda.so"

# Tesla T4 có Compute Capability 7.5
os.environ["CMAKE_CUDA_ARCHITECTURES"] = "75"

os.environ["PATH"] = (
    f"{CUDA_PATH}/bin:"
    + os.environ.get("PATH", "")
)

os.environ["LD_LIBRARY_PATH"] = (
    f"/usr/local/nvidia/lib64:{CUDA_PATH}/lib64"
)

os.environ["LIBRARY_PATH"] = (
    f"/usr/local/nvidia/lib64:{CUDA_PATH}/lib64"
)

os.environ["CMAKE_LIBRARY_PATH"] = (
    f"/usr/local/nvidia/lib64:{CUDA_PATH}/lib64"
)

print("CUDA_HOME:", os.environ["CUDA_HOME"])
print("CUDACXX:", os.environ["CUDACXX"])
print("CUDA architecture:", os.environ["CMAKE_CUDA_ARCHITECTURES"])
print("Da thiet lap CUDA cho Tesla T4")

# Backup & Restore

In [ ]:
import os, stat, urllib.request

CLOUDFLARED_PATH = "/usr/local/bin/cloudflared"
CLOUDFLARED_URL = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"

if not os.path.exists(CLOUDFLARED_PATH):
    print("Downloading cloudflared...")
    urllib.request.urlretrieve(CLOUDFLARED_URL, CLOUDFLARED_PATH)
    os.chmod(CLOUDFLARED_PATH, os.stat(CLOUDFLARED_PATH).st_mode | stat.S_IEXEC)
    print("cloudflared ready at", CLOUDFLARED_PATH)
else:
    print("cloudflared already exists at", CLOUDFLARED_PATH)

In [ ]:
import os, re, time, pathlib, subprocess, urllib.request, urllib.parse
import pty, select
from IPython.display import display, HTML

WAIT_SECONDS_SHORT = 20

def run(cmd, check=True):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check)

def tail(path, n=3000):
    p = pathlib.Path(path)
    if not p.exists():
        return ""
    return p.read_text(errors="ignore")[-n:]

def run_pty(cmd):
    """Run cmd in PTY and stream output to stdout in realtime. Return exit code."""
    master_fd, slave_fd = pty.openpty()
    proc = subprocess.Popen(
        cmd,
        stdin=slave_fd,
        stdout=slave_fd,
        stderr=slave_fd,
        close_fds=True,
    )
    os.close(slave_fd)
    try:
        while True:
            r, _, _ = select.select([master_fd], [], [], 0.2)
            if master_fd in r:
                try:
                    data = os.read(master_fd, 4096)
                except OSError:
                    break
                if not data:
                    break
                text = data.decode(errors="replace")
                print(text, end="", flush=True)
            if proc.poll() is not None:
                break
    finally:
        try:
            os.close(master_fd)
        except OSError:
            pass
    return proc.wait()

def find_token_in_log(log_path):
    text = pathlib.Path(log_path).read_text(errors="ignore")
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("{") and "access_token" in line and "refresh_token" in line:
            return line
    return None

def write_rclone_conf(token_json, client_id=None, client_secret=None):
    os.makedirs("/root/.config/rclone", exist_ok=True)
    conf_path = "/root/.config/rclone/rclone.conf"
    with open(conf_path, "w") as f:
        f.write("[" + REMOTE + "]\n")
        f.write("type = drive\n")
        f.write("scope = drive\n")
        if client_id:
            f.write("client_id = " + client_id + "\n")
        if client_secret:
            f.write("client_secret = " + client_secret + "\n")
        f.write("token = " + token_json + "\n")
    os.chmod(conf_path, 0o600)
    return conf_path

# Install rclone (required for backup whether using secret or fresh auth)
run(["apt-get", "update", "-qq"], check=False)
run(["apt-get", "install", "-y", "rclone", "wget", "fuse3", "rsync"], check=False)

# ==================================================================
# PHASE 1: GET TOKEN (from Kaggle Secret or OAuth flow)
# ==================================================================
token_json = None

# --- Load custom Google API credentials from Kaggle Secrets ---
gdrive_client_id = None
gdrive_client_secret = None
if USE_CUSTOM_GDRIVE_API and CONFIGURE_DRIVE:
    try:
        from kaggle_secrets import UserSecretsClient
        secrets_client = UserSecretsClient()
        gdrive_client_id = secrets_client.get_secret("GDRIVE_CLIENT_ID")
        gdrive_client_secret = secrets_client.get_secret("GDRIVE_CLIENT_SECRET")
        if gdrive_client_id and gdrive_client_secret:
            print(f"Loaded custom Google API credentials (client_id: ...{gdrive_client_id[-12:]})")
        else:
            print("USE_CUSTOM_GDRIVE_API=True but GDRIVE_CLIENT_ID/SECRET missing in Secrets.")
            print("Falling back to rclone default OAuth app.")
            gdrive_client_id = None
            gdrive_client_secret = None
    except Exception as e:
        print("Cannot load GDRIVE_CLIENT_ID/SECRET:", e)
        print("Falling back to rclone default OAuth app.")
        gdrive_client_id = None
        gdrive_client_secret = None

if CONFIGURE_DRIVE:
    try:
        from kaggle_secrets import UserSecretsClient
        secret_token = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")
        if secret_token:
            print("Found Kaggle Secret: RCLONE_DRIVE_TOKEN")
            token_json = secret_token

            # --- Detect stale token when switching to custom credentials ---
            if USE_CUSTOM_GDRIVE_API and gdrive_client_id:
                write_rclone_conf(token_json, gdrive_client_id, gdrive_client_secret)
                print("Testing token with custom credentials (rclone --dry-run)...")
                test = subprocess.run(
                    ["rclone", "mkdir", REMOTE + ":" + DRIVE_BACKUP_DIR + "/.__stale_test__", "--dry-run"],
                    capture_output=True, text=True, timeout=30,
                )
                err = (test.stderr or test.stdout).lower()
                if test.returncode != 0 and ("invalid_client" in err or "invalid_grant" in err):
                    print("Token was created with rclone default app (incompatible with custom credentials).")
                    print("Triggering re-auth with custom Client ID/Secret...")
                    token_json = None
                else:
                    subprocess.run(["rclone", "purge", REMOTE + ":" + DRIVE_BACKUP_DIR + "/.__stale_test__"],
                                    capture_output=True, timeout=30)
    except Exception as e:
        print("No RCLONE_DRIVE_TOKEN secret found. Need to authenticate.")

    if not token_json:
        print(subprocess.check_output(["rclone", "version"], text=True).splitlines()[0])
        print(subprocess.check_output(["cloudflared", "--version"], text=True).strip())

        subprocess.run(["pkill", "-f", "rclone authorize"], stderr=subprocess.DEVNULL)
        subprocess.run(["pkill", "-f", "cloudflared tunnel"], stderr=subprocess.DEVNULL)

        rclone_log = "/tmp/rclone_auth.log"
        cloud_log = "/tmp/cloudflared.log"
        open(rclone_log, "w").close()
        open(cloud_log, "w").close()

        print("Starting rclone authorize...")
        auth_env = os.environ.copy()
        if gdrive_client_id:
            auth_env["RCLONE_DRIVE_CLIENT_ID"] = gdrive_client_id
            auth_env["RCLONE_DRIVE_CLIENT_SECRET"] = gdrive_client_secret
            print("Using custom Google API credentials for authorize.")
        subprocess.Popen(
            ["rclone", "authorize", "drive", "--auth-no-open-browser"],
            stdout=open(rclone_log, "a"),
            stderr=subprocess.STDOUT,
            start_new_session=True,
            env=auth_env,
        )
        time.sleep(3)

        print("Starting Cloudflare Quick Tunnel for OAuth callback...")
        subprocess.Popen(
            ["cloudflared", "tunnel", "--url", "http://127.0.0.1:53682"],
            stdout=open(cloud_log, "a"),
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

        tunnel_url = None
        local_auth = None
        for _ in range(30):
            cloud_text = tail(cloud_log, 8000)
            rclone_text = tail(rclone_log, 8000)
            tunnels = re.findall(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", cloud_text)
            auths = re.findall(r"http://127\.0\.0\.1:53682[^\s]+", rclone_text)
            if tunnels:
                tunnel_url = tunnels[0]
            if auths:
                local_auth = auths[-1]
            if tunnel_url and local_auth:
                break
            time.sleep(1)

        if not tunnel_url:
            print("=== cloudflared log ===")
            print(tail(cloud_log))
            print("Cannot create tunnel for OAuth.")
        elif not local_auth:
            print("=== rclone log ===")
            print(tail(rclone_log))
            print("Cannot get rclone auth URL.")
        else:
            public_auth = local_auth.replace("http://127.0.0.1:53682", tunnel_url)
            print()
            print("=" * 90)
            print("CLICK LINK NAY DE LOGIN GOOGLE DRIVE:")
            print(public_auth)
            print("=" * 90)
            display(HTML('<a href="' + public_auth + '" target="_blank">\n  Open Google Drive Authorization\n</a>'))

            start = time.time()
            while time.time() - start < WAIT_SECONDS_SHORT:
                token_json = find_token_in_log(rclone_log)
                if token_json:
                    break
                time.sleep(1)

            if not token_json:
                print()
                print("=" * 90)
                print("SAU KHI LOGIN XONG, Google redirect ve http://127.0.0.1:53682/...")
                print("Trinh duyet se bao loi. COPY TOAN BO URL TU THANH DIA CHI")
                print("ROI PASTE XUONG DAY:")
                print("=" * 90)
                pasted = input("Paste URL: ").strip()
                if pasted and "127.0.0.1:53682" in pasted:
                    parsed = urllib.parse.urlparse(pasted)
                    try:
                        urllib.request.urlopen("http://127.0.0.1:53682/?" + parsed.query, timeout=10)
                        print("Callback forwarded. Waiting for token...")
                    except Exception as e:
                        print("Forward error:", e)
                    time.sleep(3)
                    tok = find_token_in_log(rclone_log)
                    if tok:
                        token_json = tok
                        print("Token received!")

            if token_json:
                print("TOKEN RECEIVED.")
                print()
                print("=== LUU LAI TOKEN CHO LAN SAU ===")
                print("Tao Kaggle Secret: Add-ons -> Secrets -> Add a new secret")
                print("  Label: RCLONE_DRIVE_TOKEN")
                print("  Value: (copy dong ben duoi)")
                print()
                print(token_json)
                print()
                print("=" * 60)

# ----------------------------------------------------------------
# Finalize Phase 1: write rclone.conf (covers BOTH Secret + OAuth)
# Without this, rclone has no config and Phase 2 mkdir will fail.
# ----------------------------------------------------------------
if token_json and CONFIGURE_DRIVE:
    write_rclone_conf(token_json, gdrive_client_id, gdrive_client_secret)
    if gdrive_client_id:
        print("rclone.conf written (with custom Google API credentials).")
    else:
        print("rclone.conf written (rclone default OAuth app).")

# ==================================================================
# PHASE 2: VERIFY TOKEN (catch revoke/expired BEFORE doing work)
# ==================================================================
BACKUP_OK = False
if token_json and CONFIGURE_DRIVE:
    print("\n=== Phase 2: Verify token ===")
    TEST_DIR = REMOTE + ":" + DRIVE_BACKUP_DIR + "/.__rclone_test__"
    verify = subprocess.run(
        ["rclone", "mkdir", TEST_DIR],
        capture_output=True, text=True, timeout=30,
    )
    if verify.returncode != 0:
        print("Token invalid or expired.")
        print("rclone error:", (verify.stderr or verify.stdout).strip())
        print("Re-authenticate: delete RCLONE_DRIVE_TOKEN secret and re-run this cell.")
        BACKUP_OK = False
    else:
        subprocess.run(["rclone", "purge", TEST_DIR], capture_output=True, timeout=30)
        BACKUP_OK = True
        print("Token verified.")
elif not token_json and CONFIGURE_DRIVE:
    print("No token obtained — skipping backup setup.")
else:
    print("Google Drive backup disabled — skipping all phases.")

# ==================================================================
# PHASE 3: RESTORE (Drive -> Kaggle, per source, merge, --ignore-times)
# ==================================================================
RESTORE_OK = False
if BACKUP_OK:
    print("\n=== Phase 3: Restore (Drive -> Kaggle) ===")
    restore_ok = True
    for src, sub, rules in SOURCES:
        dst = REMOTE + ":" + DRIVE_BACKUP_DIR + ("/" + sub if sub else "")
        lsl = subprocess.run(["rclone", "lsl", dst, "--max-depth", "1"],
                             capture_output=True, text=True, timeout=30)
        if lsl.returncode != 0:
            print(f"Cannot list {dst} (not found or rate limit). Treating as nothing-to-restore.")
            print("Background sync will retry later.")
            continue
        if not lsl.stdout.strip():
            print(f"{dst} is empty. Nothing to restore. OK.")
            continue
        n_entries = len(lsl.stdout.strip().splitlines())
        print(f"Found {n_entries} entries in {dst}. Restoring to {src} (merge, --ignore-times)...")
        os.makedirs(src, exist_ok=True)
        restore_cmd = ["rclone", "copy", dst, src,
                       "--ignore-times", "--stats=2s", "--stats-one-line"]
        for rule in rules:
            restore_cmd += ["--filter", rule]
        print("$", " ".join(restore_cmd))
        code = None
        for attempt in range(2):
            if attempt > 0:
                print(f"Retry in 60s (attempt {attempt + 1}/2)...")
                time.sleep(60)
            code = run_pty(restore_cmd)
            if code == 0:
                print(f"\nRestore OK: {src} (attempt {attempt + 1}).")
                break
            print(f"\nRestore attempt {attempt + 1} failed (exit {code}).")
        if code != 0:
            print(f"Restore FAILED for {src}.")
            restore_ok = False
    RESTORE_OK = restore_ok
else:
    print("Phase 2 failed - skipping Phase 3.")

# ==================================================================
# PHASE 4: INITIAL BACKUP (Kaggle -> Drive, per source)
# Only runs if BACKUP_OK AND RESTORE_OK
# ==================================================================
INITIAL_BACKUP_OK = False
if BACKUP_OK and RESTORE_OK:
    print("\n=== Phase 4: Initial backup (Kaggle -> Drive) ===")
    backup_ok = True
    for src, sub, rules in SOURCES:
        dst = REMOTE + ":" + DRIVE_BACKUP_DIR + ("/" + sub if sub else "")
        if not os.path.exists(src) or not any(pathlib.Path(src).rglob("*")):
            print(f"{src} empty or missing. Skip - background sync will catch up when data appears.")
            continue
        cmd = ["rclone", "copy", src, dst, "--stats=2s", "--stats-one-line"]
        for rule in rules:
            cmd += ["--filter", rule]
        print("$", " ".join(cmd))
        code = run_pty(cmd)
        if code == 0:
            print(f"OK: {src} -> {dst}")
        else:
            print(f"FAILED: {src} -> {dst} (exit {code})")
            print("Common causes: token expired mid-copy, quota full, network.")
            backup_ok = False
    INITIAL_BACKUP_OK = backup_ok
else:
    if not BACKUP_OK:
        print("Phase 2 (verify) failed - skipping Phase 4.")
    elif not RESTORE_OK:
        print("Phase 3 (restore) failed - skipping Phase 4.")

# ==================================================================
# PHASE 5: SYNC SCRIPT + FINAL FLAG
# ==================================================================
BACKUP_OK = BACKUP_OK and RESTORE_OK and INITIAL_BACKUP_OK

if BACKUP_OK:
    SYNC_SCRIPT = "/kaggle/working/rclone_sync.sh"
    lines = [
        "#!/bin/bash",
        "set +e",
        "INTERVAL=${1:-300}",
        "LOG=/kaggle/working/rclone_sync.log",
        "while true; do",
    ]
    for src, sub, rules in SOURCES:
        dst = REMOTE + ":" + DRIVE_BACKUP_DIR + ("/" + sub if sub else "")
        ex = " ".join('--filter "' + r + '"' for r in rules)
        lines += [
            f'  echo "[rclone_copy] $(date) {src}" >> "$LOG"',
            f'  if [ -d "{src}" ]; then',
            f'    rclone copy "{src}/" "{dst}" {ex} --stats=30s --stats-one-line --log-file "$LOG" --log-level INFO 2>> "$LOG"',
            f'    echo "[rclone_copy] $(date) {src} done rc=$?" >> "$LOG"',
            "  else",
            f'    echo "[rclone_copy] {src} not found, skip" >> "$LOG"',
            "  fi",
        ]
    lines += ['  sleep "$INTERVAL"', "done", ""]
    with open(SYNC_SCRIPT, "w") as f:
        f.write("\n".join(lines))
    os.chmod(SYNC_SCRIPT, 0o755)
    print("Created " + SYNC_SCRIPT)
    print("Note: using rclone COPY (not sync) - will NOT delete files on Drive.")
else:
    print("rclone_sync.sh NOT created - see errors above.")

print("\nHOAN TAT.")
print(f"BACKUP_OK = {BACKUP_OK}")


# Model (symlink from Kaggle input)

In [ ]:
from pathlib import Path

SOURCE = Path(
    "/kaggle/input/models/buianhtruc/strata-qwen3-8-flash-next-gsq-rco-iq3-s/gguf/default/1" #SOURCE your kaggle input 
)
TARGET = Path("/root/Strata-data/models/IQ3_S")

if not SOURCE.exists():
    raise FileNotFoundError(f"Không tìm thấy nguồn: {SOURCE}")

# Tạo thư mục đích
TARGET.mkdir(parents=True, exist_ok=True)

# Xóa các symlink cũ trong thư mục đích
for item in TARGET.iterdir():
    if item.is_symlink():
        item.unlink()

# Liên kết toàn bộ file GGUF
gguf_files = sorted(SOURCE.glob("*.gguf")) + sorted(SOURCE.glob("*.done"))

if not gguf_files:
    raise FileNotFoundError(f"Không tìm thấy file GGUF trong: {SOURCE}")

for source_file in gguf_files:
    target_file = TARGET / source_file.name
    target_file.symlink_to(source_file)
    print(f"Đã gán: {target_file.name}")

print(f"\nTìm thấy và gán {len(gguf_files)} file GGUF")
print(f"Thư mục model: {TARGET}")

# Install Strata

In [ ]:
import shlex, subprocess
from kaggle_secrets import UserSecretsClient

# Doc API key tu Kaggle Secret (Add-ons -> Secrets -> Label: API_KEY).
# Khong co secret thi bo qua --api-key (server chay khong can key).
try:
    API_KEY = UserSecretsClient().get_secret("API_KEY")
except Exception:
    API_KEY = None

cmd = ["sh", "/root/Strata/setup.sh", "--setup", "--model", "IQ3_S", "--gpus", "0,1",
       "--context", "524288", "--kv", "q4_0",
       "--vision", "yes", "--experimental-speed-projection", "on",
       "--no-start", "--yes"]
if API_KEY:
    cmd += ["--api-key", API_KEY]
    print("Using API key from Kaggle Secret: API_KEY")
else:
    print("No API_KEY secret found - skipping --api-key (server runs without a key).")

print("$ " + " ".join("****" if API_KEY and a == API_KEY else shlex.quote(a) for a in cmd))
rc = subprocess.call(cmd)
print("setup exit code:", rc)

# Start Strata

In [ ]:
import os
import subprocess
import time
import urllib.request

# Resolve Strata port (set in the Setup config cell; fallback 8080)
STRATA_PORT = str(globals().get("STRATA_PORT", "8080"))
STRATA_URL = f"http://127.0.0.1:{STRATA_PORT}"
TUNNEL_TOKEN_LABEL = str(globals().get("TUNNEL_TOKEN_LABEL", "CLOUDFLARE_TUNNEL_TOKEN"))

# =====================
# STOP OLD PROCESSES
# =====================
for pattern in ["serve/server.py", "cloudflared", "ngrok"]:
    subprocess.run(
        ["pkill", "-f", pattern],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

time.sleep(2)

# =====================
# START STRATA
# =====================
import glob

STRATA_DIR = "/root/Strata"
cfgs = [p for p in sorted(glob.glob(STRATA_DIR + "/strata-*.json"))
        if "shared-settings" not in p]
if not cfgs:
    print("No strata-*.json config found. Run the Install Strata cell first.")
else:
    cfg_path = cfgs[0]
    print("Using config:", cfg_path)

    log_path = "/kaggle/working/strata_server.log"
    log_file = open(log_path, "w")
    strata_process = subprocess.Popen(
        [STRATA_DIR + "/.venv/bin/python", "serve/server.py",
         "--engine", "strata", "--config", cfg_path, "--port", str(STRATA_PORT)],
        cwd=STRATA_DIR,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=os.environ.copy(),
    )
    log_file.close()

    print("Strata server started (PID:", strata_process.pid, ")")
    print(f"API (OpenAI): {STRATA_URL}/v1  (any API key; model name: anything)")
    print("Log:", log_path)

    # =====================
    # HEALTH CHECK (loading the model can take 1-3 minutes)
    # =====================
    HEALTH_OK = False
    for i in range(60):
        try:
            health = urllib.request.urlopen(
                f"{STRATA_URL}/health", timeout=10
            ).read().decode()
            print("Health check OK:", health)
            HEALTH_OK = True
            break
        except Exception:
            print(f"  waiting for server... ({i + 1}/60)")
            time.sleep(5)
    if not HEALTH_OK:
        print("Health check FAILED. Log tail:")
        print(open(log_path).read()[-2000:])

# =====================
# BACKGROUND SYNC (only if initial setup succeeded)
# =====================
try:
    BACKUP_OK
except NameError:
    BACKUP_OK = False

if not BACKUP_OK:
    print("=== Initial setup FAILED (see Cell 10 output). Skipping background sync. ===")
elif os.path.exists("/kaggle/working/rclone_sync.sh"):
    sync_proc = subprocess.Popen(
        ["bash", "/kaggle/working/rclone_sync.sh", str(SYNC_INTERVAL)],
        start_new_session=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        stdin=subprocess.DEVNULL,
        close_fds=True,
    )
    print(f"Background rclone sync started (PID: {sync_proc.pid})")
    print("Check log: !tail -f /kaggle/working/rclone_sync.log")

    # Peek first sync run after 35s
    time.sleep(35)
    log_path_sync = "/kaggle/working/rclone_sync.log"
    if os.path.exists(log_path_sync):
        print("--- rclone_sync.log (last 1500 chars) ---")
        print(open(log_path_sync).read()[-1500:])
else:
    print("Sync script not found. Run the Google Drive cell (Cell 10) first.")

# =====================
# TUNNEL
# =====================
if TUNNEL_MODE == "quick":
    print("Starting Cloudflare Quick Tunnel...")
    tunnel_process = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", STRATA_URL],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

elif TUNNEL_MODE == "named":
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    try:
        token = secrets.get_secret(TUNNEL_TOKEN_LABEL)
    except Exception:
        token = None
    if not token and TUNNEL_TOKEN_LABEL != "CLOUDFLARE_TUNNEL_TOKEN":
        print(f"Khong tim thay {TUNNEL_TOKEN_LABEL}, fallback ve CLOUDFLARE_TUNNEL_TOKEN...")
        try:
            token = secrets.get_secret("CLOUDFLARE_TUNNEL_TOKEN")
        except Exception as e:
            print("Error getting CLOUDFLARE_TUNNEL_TOKEN:", e)
            token = None
    if not token:
        print("Set it in Add-ons -> Secrets and re-run.")
    if token:
        print(f"Starting Cloudflare Named Tunnel -> {STRATA_URL} ...")
        tunnel_process = subprocess.Popen(
            ["cloudflared", "tunnel", "run", "--token", token],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        print("LUU Y: Named tunnel dung ingress tren Cloudflare dashboard.")
        print(f"Vao dashboard tunnel -> Public hostname -> Service -> sua thanh http://127.0.0.1:{STRATA_PORT}")
    else:
        print("Cannot start named tunnel without token.")
        tunnel_process = None

elif TUNNEL_MODE == "ngrok":
    import zipfile, shutil, stat
    from kaggle_secrets import UserSecretsClient

    NGROK_PATH = "/kaggle/working/ngrok"
    NGROK_ZIP = "/kaggle/working/ngrok.zip"
    NGROK_URL = "https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip"

    if not os.path.exists(NGROK_PATH):
        print("Downloading ngrok...")
        urllib.request.urlretrieve(NGROK_URL, NGROK_ZIP)
        with zipfile.ZipFile(NGROK_ZIP, "r") as z:
            z.extractall("/kaggle/working")
        os.chmod(NGROK_PATH, os.stat(NGROK_PATH).st_mode | stat.S_IEXEC)

    secrets = UserSecretsClient()
    try:
        NGROK_AUTHTOKEN = secrets.get_secret("NGROK_AUTHTOKEN")
        NGROK_DOMAIN = secrets.get_secret("NGROK_DOMAIN")
    except Exception as e:
        print("Error getting ngrok secrets:", e)
        print("Set NGROK_AUTHTOKEN and NGROK_DOMAIN in Add-ons -> Secrets.")
        NGROK_AUTHTOKEN = None

    if NGROK_AUTHTOKEN:
        subprocess.run([NGROK_PATH, "config", "add-authtoken", NGROK_AUTHTOKEN], check=False)
        print(f"Starting ngrok... https://{NGROK_DOMAIN}")
        tunnel_process = subprocess.Popen(
            [NGROK_PATH, "http", f"--domain={NGROK_DOMAIN}", STRATA_PORT],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
    else:
        print("Cannot start ngrok without auth token.")
        tunnel_process = None

# =====================
# KEEP TUNNEL ALIVE
# =====================
if tunnel_process:
    for line in tunnel_process.stdout:
        print(line, end="")
else:
    print(f"No tunnel process. Local access only: {STRATA_URL}")
    print("Cell will exit immediately. Rerun with a tunnel mode if needed.")
